<a href="https://colab.research.google.com/github/ryouy/GraduationThesis/blob/main/1015/white_bear_and_zhou_pilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# White Bear Effect — Colab Pilot: Mann + Zhou

**Mann et al. (2025) に着想を得た小規模追試 + 探索的な置換・連想実験**

- モデル: `Qwen/Qwen2.5-0.5B-Instruct`（公開・軽量）
- 主指標: 指定概念が**次に生成される条件付き対数確率**（log probability）
- 比較: neutral / suppression / affirm / replacement × distractor length
- 追加: unrelated cue vs associative cue

> **注意**: 原論文の完全な実験再現ではありません。モデル、刺激、条件設計をColab向けに簡略化しました。結果は研究用の予備実験として解釈してください。

元論文: https://arxiv.org/abs/2511.12381

GPUランタイム推奨（T4で可）。Hugging Faceへのログインは不要です。

## 1. 環境とモデル

In [ ]:
%pip -q install "transformers>=4.46,<5" "accelerate>=1.0,<2" pandas matplotlib seaborn tqdm

In [ ]:
import os, gc, random, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
MODEL_ID="Qwen/Qwen2.5-0.5B-Instruct"
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
print("device:",DEVICE, "torch:",torch.__version__)
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=False)
model=AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16 if DEVICE=="cuda" else torch.float32,
    low_cpu_mem_usage=True, trust_remote_code=False
).to(DEVICE).eval()
print("model loaded", MODEL_ID)


## 2. スコア計算

候補文字列 `" banana"` の**全トークン**について、文脈を与えたあとの対数確率を加算します。候補が複数トークンになっても正しく計算できます。ただし長さの異なる語の生の対数確率は直接比較しません。

この値は「内部概念の活性量」そのものではありません。出力ヘッドを通した候補文字列の生成尤度です。

In [ ]:
@torch.inference_mode()
def continuation_logprob(context: str, continuation: str) -> float:
    # Add candidate by token IDs, avoiding ambiguous special-token additions.
    context_ids=tokenizer(context, add_special_tokens=False).input_ids
    candidate_ids=tokenizer(continuation, add_special_tokens=False).input_ids
    if not context_ids or not candidate_ids: raise ValueError("Empty context/candidate")
    full_ids=context_ids+candidate_ids
    x=torch.tensor([full_ids],dtype=torch.long,device=DEVICE)
    # For each continuation token at index k, use logits at k-1.
    logits=model(input_ids=x).logits[0]
    logp=torch.log_softmax(logits.float(),dim=-1)
    start=len(context_ids)
    return float(sum(logp[k-1, full_ids[k]].item() for k in range(start,len(full_ids))))

print('example',continuation_logprob('The word is',' banana'))


## 3. 固定文脈と条件

主実験ではすべての条件に対象語を**同じ回数**含め、指示極性だけを操作します。長さの異なる妨害文を後ろに追加し、最後に `The word is` という共通文脈を置きます。

**解釈上の制約**: プロンプトの自然さや肯定/否定文の長さは完全には一致しておらず、「因果的な抑制機構の発見」とは言えません。

In [ ]:
CONCEPTS=["banana","elephant","tiger","piano","coffee","ocean","castle","bicycle"]
REPLACEMENTS={"banana":"apple","elephant":"giraffe","tiger":"lion","piano":"violin","coffee":"tea","ocean":"river","castle":"palace","bicycle":"scooter"}
DISTRACTOR="The train arrived at the station on time. The passengers checked their tickets. "
LEVELS=[0,1,3,6]

def make_prompt(x, condition, n_distractors):
    y=REPLACEMENTS[x]
    templates={
      'neutral':f'The topic word is {x}.',
      'affirm':f'Think about {x}.',
      'suppression':f'Do not think about {x}.',
      'replacement':f'Think about {y} instead of {x}.',
    }
    return templates[condition]+'\n'+DISTRACTOR*n_distractors+'\nComplete the phrase with a single word.\nThe word is'

print(make_prompt('banana','suppression',1))


## 4. 先行研究に近い予備実験: distractor length × negation

In [ ]:
rows=[]
for x in tqdm(CONCEPTS,desc='concepts'):
    for n_d in LEVELS:
        for condition in ('neutral','affirm','suppression','replacement'):
            p=make_prompt(x,condition,n_d)
            lp=continuation_logprob(p,' '+x)
            rows.append({'concept':x,'condition':condition,'distractor_blocks':n_d,
                         'logprob':lp,'probability':float(math.exp(lp)),
                         'prompt':p})
df=pd.DataFrame(rows)
assert len(df)==len(CONCEPTS)*len(LEVELS)*4
display(df.head())


In [ ]:
# Within-concept paired differences, not raw comparison across concept spellings
wide=df.pivot_table(index=['concept','distractor_blocks'],columns='condition',values='logprob').reset_index()
wide['suppression_minus_neutral']=wide['suppression']-wide['neutral']
wide['suppression_minus_affirm']=wide['suppression']-wide['affirm']
wide['replacement_minus_suppression']=wide['replacement']-wide['suppression']
display(wide.groupby('distractor_blocks')[['suppression_minus_neutral','suppression_minus_affirm','replacement_minus_suppression']].agg(['mean','std']).round(3))

fig,ax=plt.subplots(figsize=(8,5))
for metric,label in [('suppression_minus_neutral','Suppression - Neutral'),('suppression_minus_affirm','Suppression - Affirm'),('replacement_minus_suppression','Replacement - Suppression')]:
    g=wide.groupby('distractor_blocks')[metric]
    mean=g.mean().reindex(LEVELS); sem=g.sem().reindex(LEVELS).fillna(0)
    ax.errorbar(LEVELS,mean,yerr=sem,marker='o',capsize=3,label=label)
ax.axhline(0,color='gray',linewidth=1,linestyle='--')
ax.set(xlabel='Distractor blocks',ylabel='Paired delta log P(target)',title='Pilot: suppression, replacement and distractor length')
ax.legend();plt.tight_layout();plt.savefig('pilot_distractor_effect.png',dpi=180);plt.show()


## 5. 追加実験（独自性候補）: 連想的手がかり

この実験はMann et al.の**厳密な再現ではなく、新規の探索的拡張**です。各対象に関連する手がかりと、無関係な手がかりを比較します。ここで使う関連質問は少数の手作業例で、正答や連想の強さが統制されていません。

In [ ]:
CUES={
'banana':('What fruit is commonly associated with a cartoon monkey?','What instrument is played using black and white keys?'),
'elephant':('What large animal has a trunk?','What hot drink is made from roasted beans?'),
'tiger':('Which striped big cat lives in Asia?','What object has pedals and two wheels?'),
'piano':('Which keyboard instrument is common in classical music?','What animal is known for a long trunk?'),
'coffee':('What drink is made from roasted beans?','What building has towers and stone walls?'),
'ocean':('What large saltwater body separates continents?','What instrument has strings and a bow?'),
'castle':('What medieval building often has towers and moats?','What fruit is usually yellow when ripe?'),
'bicycle':('What two-wheeled vehicle has pedals?','What is the name for a large body of saltwater?'),
}

def make_cue_prompt(x,condition,cue_type):
    # a common continuation format makes comparisons possible
    base=make_prompt(x,condition,0).split('\nComplete the phrase')[0]
    cue=CUES[x][0 if cue_type=='related' else 1]
    return base+'\n'+cue+'\nAnswer with one word.\nAnswer:'

cue_rows=[]
for x in tqdm(CONCEPTS,desc='cue experiments'):
    for cond in ('neutral','affirm','suppression','replacement'):
        for cue in ('related','unrelated'):
            p=make_cue_prompt(x,cond,cue)
            cue_rows.append({'concept':x,'condition':cond,'cue':cue,
                             'logprob':continuation_logprob(p,' '+x),'prompt':p})
cue_df=pd.DataFrame(cue_rows)
cue_wide=cue_df.pivot(index=['concept','condition'],columns='cue',values='logprob').reset_index()
cue_wide['related_minus_unrelated']=cue_wide['related']-cue_wide['unrelated']
display(cue_wide.groupby('condition')['related_minus_unrelated'].agg(['mean','std','count']).round(3))
fig,ax=plt.subplots(figsize=(8,4))
cues=cue_wide.groupby('condition')['related_minus_unrelated'].agg(['mean','sem']).reindex(['neutral','affirm','suppression','replacement'])
ax.bar(cues.index,cues['mean'],yerr=cues['sem'],capsize=4)
ax.axhline(0,color='gray',linewidth=1)
ax.set(ylabel='Related - Unrelated: delta log P(target)',title='Associative cue effect (exploratory)')
plt.tight_layout();plt.savefig('pilot_associative_cue.png',dpi=180);plt.show()


## 6. 単語が実際に生成されるか（定性的な参考）

自由生成の結果は確率指標とは別です。生成文に対象単語が含まれないことは、内部で表現されないことを意味しません。

In [ ]:
@torch.inference_mode()
def generate_answer(prompt,max_new_tokens=24):
    enc=tokenizer(prompt,return_tensors='pt').to(DEVICE)
    out=model.generate(**enc,max_new_tokens=max_new_tokens,do_sample=False,
                       pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0,enc['input_ids'].shape[1]:],skip_special_tokens=True).strip()

for condition in ('neutral','affirm','suppression','replacement'):
    p=make_cue_prompt('banana',condition,'related')
    print(condition,':',repr(generate_answer(p)))


## 7. 保存と研究上の注意

- CSVに各刺激・各条件の測定値を保存します。出力グラフもPNGで保存します。
- ポスターでは「予備実験」「Mann et al. (2025) に着想を得た部分追試」と明記してください。
- 観測するlog-probabilityと概念のhidden-state activationを混同しないでください。
- 元論文の数値との比較には、同じモデル・同じ対象語集合・同じ妨害テキスト・同じ候補確率算出規則が必要です。
- 研究を本格化するなら複数seed、刺激のランダム化、モデル比較、事前登録した統計検定、hidden-state probe / causal interventionを追加します。

In [ ]:
import json, zipfile
meta={'paper':'Mann et al. 2025 inspired pilot (NOT exact replication)',
      'model':MODEL_ID,'seed':SEED,'torch':torch.__version__,
      'concepts':CONCEPTS,'levels':LEVELS}
df.to_csv('pilot_distractor_raw.csv',index=False)
wide.to_csv('pilot_distractor_paired.csv',index=False)
cue_df.to_csv('pilot_associative_raw.csv',index=False)
cue_wide.to_csv('pilot_associative_paired.csv',index=False)
with open('pilot_metadata.json','w') as f:json.dump(meta,f,indent=2)
filenames=['pilot_distractor_raw.csv','pilot_distractor_paired.csv','pilot_associative_raw.csv','pilot_associative_paired.csv','pilot_distractor_effect.png','pilot_associative_cue.png','pilot_metadata.json']
with zipfile.ZipFile('white_bear_pilot_results.zip','w') as z:
    for fn in filenames:z.write(fn)
print('Saved: white_bear_pilot_results.zip')
# Optional: uncomment in Colab
# from google.colab import files; files.download('white_bear_pilot_results.zip')


# 8. Zhou et al. (2026) の簡易追試：肯定・否定の正答率と層別表現

**原論文**: [How Language Models Process Negation (ICML 2026)](https://proceedings.mlr.press/v306/zhou26bq.html) / [著者の公開コード・データ](https://github.com/Ja1Zhou/LM_Negation)

原論文では、否定質問への誤答と、正しく否定を扱う内部機構が共存すること、特に後段のAttentionによるショートカット、**suppression**と**construction**の2機構（construction優勢）を報告しています。

この節は**原論文の完全再現ではありません**。論文の公開データ（648ペア）、Mistral-7B / Llama-3.1-8B、Attention Head ablationや因果パッチングは使用していません。代わりに、T4でも軽く回るQwen2.5-0.5Bを使った**操作確認・簡易追試**です。

- A: 「Xである」「Xではない」の問題で、正しい候補のスコア・選択精度を測る。
- B: 中間層の状態から候補スコアを可視化する（**logit lens的な観測**。内部回路の因果的同定ではない）。
- 重要: **否定文は単に`X`を弱めるとは限りません**。例えば `not gas` は `liquid` や `solid` などの概念を支持する可能性があります。


In [ ]:
# 小さな3択の「性質の否定」課題。正答はルールに基づく。
# 同じ三択で、問題文が変わるたびに正解が変わるようにする。
# 「not X」のみでは複数正解となるため、追加制約を設けて正解を一意にしています。
ZHOU_CASES = [
  ('gas', 'liquid', 'solid', 'gas'),
  ('gas', 'liquid', 'solid', 'liquid'),
  ('gas', 'liquid', 'solid', 'solid'),
  ('red', 'blue', 'green', 'red'),
  ('red', 'blue', 'green', 'blue'),
  ('red', 'blue', 'green', 'green'),
  ('metal', 'wood', 'plastic', 'metal'),
  ('metal', 'wood', 'plastic', 'wood'),
  ('metal', 'wood', 'plastic', 'plastic'),
]
# negation: "not first AND not second" => third ; positive: "is third" => third
# 候補名を明記する方式のため語彙バイアスが残ります。
def zhou_prompts(a,b,c,correct):
    opts=f'A) {a}\nB) {b}\nC) {c}'
    if correct == a:
        pos=f'The item is {a}.'
        neg=f'The item is not {b} and not {c}.'
    elif correct == b:
        pos=f'The item is {b}.'
        neg=f'The item is not {a} and not {c}.'
    else:
        pos=f'The item is {c}.'
        neg=f'The item is not {a} and not {b}.'
    def wrap(statement):
        return f'Choose exactly one option based ONLY on the statement.\n{statement}\n{opts}\nAnswer:'
    return {'positive':wrap(pos),'negative':wrap(neg)}
print(zhou_prompts(*ZHOU_CASES[0])['negative'])

## 8A. 出力側の検証（正答率・確率差）

3択のラベル ` A`, ` B`, ` C` の継続対数尤度を比較します。候補間でsoftmaxした値は**三択内で正規化した相対値**であり、モデルが全語彙でこの確率で回答するという意味ではありません。

In [ ]:
import torch.nn.functional as F
Z_LABELS=[' A',' B',' C']
zh_rows=[]
for case_id,(a,b,c,correct) in enumerate(tqdm(ZHOU_CASES,desc='Zhou mini behavior')):
    gold=[' A',' B',' C'][[a,b,c].index(correct)]
    for polarity,prompt in zhou_prompts(a,b,c,correct).items():
        scores=np.array([continuation_logprob(prompt,label) for label in Z_LABELS])
        probs=np.exp(scores-scores.max()); probs=probs/probs.sum()
        pred=Z_LABELS[int(probs.argmax())]
        zh_rows.append({'case_id':case_id,'polarity':polarity,'correct_option':gold.strip(),
           'pred_option':pred.strip(),'is_correct':int(pred==gold),
           'p_gold_conditional':float(probs[Z_LABELS.index(gold)]),
           'p_A':float(probs[0]),'p_B':float(probs[1]),'p_C':float(probs[2]),
           'prompt':prompt})
zh_df=pd.DataFrame(zh_rows)
print(zh_df.groupby('polarity')[['is_correct','p_gold_conditional']].mean().round(3))
display(zh_df[['case_id','polarity','correct_option','pred_option','p_gold_conditional']])
zh_df.to_csv('zhou_mini_behavior.csv',index=False)
fig,ax=plt.subplots(figsize=(6,4))
zh_df.groupby('polarity')['is_correct'].mean().reindex(['positive','negative']).plot.bar(ax=ax)
ax.set_ylim(0,1);ax.set_ylabel('Accuracy');ax.set_title('Positive vs Negative (mini pilot)')
plt.tight_layout();plt.savefig('zhou_mini_accuracy.png',dpi=160);plt.show()

## 8B. 層ごとの候補支持（exploratory logit lens）

最後の `Answer:` トークンのhidden stateを各層から取得し、**最終layer normとLM headを各層へ共通適用**して、A/B/Cの相対支持を観測します。モデルの中間層は必ずしも語彙空間に直接整列していないため、初期層スコアの解釈には注意してください。

**厳密な因果検証ではない**ため、この結果だけで「suppression回路」や「construction回路」の存在を結論づけられません。

In [ ]:
# Qwen2.5 では " A" などが単一tokenかチェックする。
# multi-tokenだと単一ステップのlogit lensは無効なので、ここでは明示的に止める。
label_ids=[tokenizer.encode(v,add_special_tokens=False) for v in Z_LABELS]
print('label token IDs:',label_ids)
if not all(len(v)==1 for v in label_ids):
    print('NOTE: multi-token labels found; using single-token labels 1,2,3 if possible.')
    alt=[' 1',' 2',' 3']
    ids_alt=[tokenizer.encode(v,add_special_tokens=False) for v in alt]
    if not all(len(v)==1 for v in ids_alt):
        raise RuntimeError('No single-token option labels available; please use alternative labels.')
    Z_LENS_LABELS=alt
    # Only lens prompts are remapped: use original prompt with options switched to numerals.
    def lens_prompt(prompt):
        return prompt.replace('A) ','1) ').replace('B) ','2) ').replace('C) ','3) ')
else:
    Z_LENS_LABELS=Z_LABELS
    def lens_prompt(prompt): return prompt
Z_LENS_IDS=[tokenizer.encode(x,add_special_tokens=False)[0] for x in Z_LENS_LABELS]

@torch.inference_mode()
def layer_option_probs(prompt):
    x=tokenizer(lens_prompt(prompt),return_tensors='pt',add_special_tokens=False).to(DEVICE)
    out=model(**x,output_hidden_states=True,return_dict=True)
    results=[]
    for h in out.hidden_states:
        state=h[:,-1,:]
        normed=model.model.norm(state)
        logits=model.lm_head(normed)[0, Z_LENS_IDS].float()
        results.append(torch.softmax(logits,dim=-1).cpu().numpy())
    return np.stack(results)

# 典型例：gas / liquid / solid の 'gas' が正解になるケース。
ex=ZHOU_CASES[0]
pair=zhou_prompts(*ex)
pp=layer_option_probs(pair['positive'])
pn=layer_option_probs(pair['negative'])
fig,axes=plt.subplots(1,2,figsize=(12,4),sharey=True)
for ax,m,title in zip(axes,[pp,pn],['Positive','Negative']):
    for j,name in enumerate([ex[0],ex[1],ex[2]]):
        ax.plot(range(len(m)),m[:,j],label=name)
    ax.set_xlabel('Hidden-state index (0=embedding)')
    ax.set_title(title);ax.set_ylim(0,1);ax.legend()
axes[0].set_ylabel('Normalized option support (logit lens)')
plt.tight_layout();plt.savefig('zhou_mini_layer_lens.png',dpi=160);plt.show()
print('Rows=layers; columns=',[ex[0],ex[1],ex[2]])
print('final positive:',pp[-1].round(3),'final negative:',pn[-1].round(3))

## 8C. どう解釈するか

1. `negative` の正答率が `positive` より低ければ、否定の処理が相対的に難しいという**行動レベル**の結果になります（小標本なので確証ではありません）。
2. 否定条件で「否定された選択肢」の確率が落ち、残る選択肢の支持が上がれば、**結果の傾向**は構成的な説明と整合します。しかし、それが構成機構によると特定したことにはなりません。
3. 中間層では正解が優勢なのに最終層で誤答になる例があれば、Zhou et al.のいう「内部情報と最終回答の乖離」を調べる**次の候補**です。後段Attentionが原因かはhead ablation等で別途確認が必要です。
4. Zhou et al.を厳密に追試する場合は、[著者のコードと公開データ](https://github.com/Ja1Zhou/LM_Negation)の `scripts/repro/` を使用し、同じモデル・解析介入を再現してください（7B〜8B級で計算資源が必要です）。

CSV: `zhou_mini_behavior.csv` / 図: `zhou_mini_accuracy.png`, `zhou_mini_layer_lens.png` 。
